In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes per rules
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Ensure 'value' columns are numeric where present
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# 1) metric.csv summary: group by (cmdb_id, kpi_name) -> count and global P95 of 'value'
metric_summary = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(count=('value', 'count'),
         p95_value=('value', lambda s: float(s.quantile(0.95)) if s.notna().any() else float('nan')))
    .sort_values(['count', 'cmdb_id', 'kpi_name'], ascending=[False, True, True])
    .head(50)
)

# 2) trace.csv: distinct (cmdb_id, trace_name) pairs with counts
trace_summary = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(count=('value', 'count'))
    .sort_values('count', ascending=False)
    .head(50)
)

# 3) log.csv: distinct (cmdb_id, log_name) pairs with counts
log_summary = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(count=('value', 'count'))
    .sort_values('count', ascending=False)
    .head(50)
)

# 4) error_logs.csv: earliest 20 rows by timestamp with selected columns
error_logs_sample = (
    error_df
    .sort_values('timestamp', ascending=True)
    .loc[:, ['timestamp', 'cmdb_id', 'message']]
    .head(20)
)

# Display compact results (IPython will show these variables)
metric_summary, trace_summary, log_summary, error_logs_sample

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry files:\n"
    "- metric.csv: Per (cmdb_id, kpi_name) global summaries computed (count and P95). "
    "Most metric series have 25 samples each in the top-50 list. Examples: adservice P95s include cpu ~0.7569, "
    "diskio ~74245.71, latency-50 ~0.00309, latency-90 ~0.00476, mem ~1.0656e8. productcatalogservice CPU P95 ~19.82; "
    "paymentservice socket P95 = 3.0. Global P95s were computed over the entire metric series (no time filtering).\n"
    "- trace.csv: Distinct (cmdb_id, trace_name) pairs listed; most pairs have 25 samples (e.g., many checkoutservice trace metrics).\n"
    "- log.csv: Distinct (cmdb_id, log_name) pairs listed; log.error_count and log.row_count appear for most services with 25 samples. "
    "redis has only 5 samples for its log series.\n"
    "- error_logs.csv: No rows returned (empty result set) — no error log rows available to display.\n\n"
    "Next steps: filter these series by the incident time window and inspect the KPIs/traces/logs whose values exceed the global P95 thresholds to identify likely root-cause components."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  count     p95_value
0               adservice         cpu     25  7.568786e-01
1               adservice      diskio     25  7.424571e+04
2               adservice  latency-50     25  3.086157e-03
3               adservice  latency-90     25  4.762612e-03
4               adservice         mem     25  1.065592e+08
..                    ...         ...    ...           ...
45         paymentservice      socket     25  3.000000e+00
46         paymentservice    workload     25  1.090240e+00
47  productcatalogservice         cpu     25  1.982458e+01
48  productcatalogservice  latency-50     25  3.794207e-03
49  productcatalogservice  latency-90     25  8.185826e-02

[50 rows x 4 columns],             cmdb_id                                trace_name  count
0   checkoutservice  trace.from_checkoutservice.duration_mean     25
1   checkoutservice   trace.from_checkoutservice.duration_p95     25
2   checkoutservice     trace.from_checkoutservice.error_rate     25
3   checkoutservice      trace.from_checkoutservice.row_count     25
4   checkoutservice  trace.from_frontendservice.duration_mean     25
..              ...                                       ...    ...
45     emailservice   trace.from_checkoutservice.duration_p95     25
46     emailservice     trace.from_checkoutservice.error_rate     25
47     emailservice      trace.from_checkoutservice.row_count     25
48     emailservice             trace.from_root.duration_mean     25
49     emailservice              trace.from_root.duration_p95     25

[50 rows x 3 columns],                   cmdb_id         log_name  count
0               adservice  log.error_count     25
1               adservice    log.row_count     25
2             cartservice  log.error_count     25
3             cartservice    log.row_count     25
4         checkoutservice  log.error_count     25
5         checkoutservice    log.row_count     25
6         currencyservice  log.error_count     25
7         currencyservice    log.row_count     25
8            emailservice  log.error_count     25
9            emailservice    log.row_count     25
10               frontend  log.error_count     25
11               frontend    log.row_count     25
12         paymentservice  log.error_count     25
13         paymentservice    log.row_count     25
14  recommendationservice  log.error_count     25
15  recommendationservice    log.row_count     25
19        shippingservice    log.row_count     25
18        shippingservice  log.error_count     25
17                  redis    log.row_count      5
16                  redis  log.error_count      5, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing metric_df if available; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Ensure correct dtypes
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using entire series
thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(global_P95=('value', lambda s: float(s.quantile(0.95)) if s.notna().any() else float('nan')),
         global_P05=('value', lambda s: float(s.quantile(0.05)) if s.notna().any() else float('nan')),
         total_count=('value', 'count'))
)

# 2) Filter metric rows to the incident window (UTC inclusive)
start = pd.to_datetime('2024-01-17 10:05:48', utc=True)
end = pd.to_datetime('2024-01-17 10:35:48', utc=True)

window_df = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# If window is empty, prepare empty result
if window_df.shape[0] == 0:
    result = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','window_count','high_count','low_count',
        'earliest_high_ts','earliest_low_ts','max_value_in_window','min_value_in_window',
        'global_P95','global_P05'
    ])
else:
    # Merge thresholds into window rows so comparisons use precomputed global thresholds
    merged = window_df.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

    # Group and compute required stats
    rows = []
    grouped = merged.groupby(['cmdb_id', 'kpi_name'], sort=False)
    for (cmdb, kpi), g in grouped:
        vals = g['value']
        gp95 = float(g['global_P95'].iloc[0])
        gp05 = float(g['global_P05'].iloc[0])

        window_count = int(vals.count())
        high_mask = vals >= gp95
        low_mask = vals <= gp05
        high_count = int(high_mask.sum())
        low_count = int(low_mask.sum())

        if high_count > 0:
            earliest_high_ts = g.loc[high_mask, 'timestamp'].min().isoformat()
        else:
            earliest_high_ts = None

        if low_count > 0:
            earliest_low_ts = g.loc[low_mask, 'timestamp'].min().isoformat()
        else:
            earliest_low_ts = None

        max_value_in_window = float(vals.max()) if vals.notna().any() else float('nan')
        min_value_in_window = float(vals.min()) if vals.notna().any() else float('nan')

        rows.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'window_count': window_count,
            'high_count': high_count,
            'low_count': low_count,
            'earliest_high_ts': earliest_high_ts,
            'earliest_low_ts': earliest_low_ts,
            'max_value_in_window': max_value_in_window,
            'min_value_in_window': min_value_in_window,
            'global_P95': gp95,
            'global_P05': gp05
        })

    result = pd.DataFrame(rows)

    # 4) Keep only rows with window_count>0, sort by severity then window_count, limit to top 20
    result = result[result['window_count'] > 0].copy()
    result['severity'] = result[['high_count', 'low_count']].max(axis=1)
    result = result.sort_values(['severity', 'window_count'], ascending=[False, False]).drop(columns=['severity'])

# Display compact result (top 20)
result.head(20)

```
Out[2]:
```
Summary of findings (window 2024-01-17 10:05:48 → 10:35:48 UTC):

- Many services' "socket" (and some other) metrics are constant across the window so their global P05 == P95 == value, causing both high_count and low_count = 25 (not informative by itself).
- Notable, actionable signals:
  - frontend — socket: window_count=25, high_count=22, earliest_high=2024-01-17T10:11:00Z, max=25.8, min=24.0 (global P95=25.0). This indicates sustained elevated socket values starting ~10:11 UTC.
  - productcatalogservice — socket: window_count=25, high_count=11, low_count=13, earliest_high=2024-01-17T10:22:00Z, earliest_low=2024-01-17T10:08:00Z (max=6, min=4). Mixed behavior with later highs.
  - redis — diskio: window_count=25, high_count=2 (earliest_high=2024-01-17T10:20:00Z), low_count=19 (earliest_low=2024-01-17T10:08:00Z), max ~22549, min 0 (global P95 ~21888). Short disk I/O spikes around 10:20 UTC.
  - emailservice — latency-50 and latency-90: each window_count=25 with high_count=2 (earliest_high=2024-01-17T10:25:00Z) and many low_count entries (earliest_low=10:08). Small number of higher-latency samples at ~10:25 UTC.
  - adservice — cpu: window_count=25, high_count=2 (earliest_high=2024-01-17T10:16:00Z), low_count=2 (earliest_low=2024-01-17T10:24:00Z); modest CPU excursions around 10:16 UTC.

Interpretation / next steps:
- The most suspicious sustained signal is frontend.socket (many samples above the global P95 starting ~10:11 UTC). Investigate frontend socket resource/connection behavior, related traces and logs.
- Investigate productcatalogservice.socket for its mixed behavior and redis disk I/O for short high spikes (~10:20 UTC).
- Check traces/logs around the earliest_high timestamps (10:11–10:25 UTC) for frontend, productcatalogservice, redis, and emailservice to narrow root cause.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  window_count  high_count  low_count           earliest_high_ts            earliest_low_ts  max_value_in_window  min_value_in_window    global_P95    global_P05
5               adservice      socket            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         4.000000e+00         4.000000e+00  4.000000e+00  4.000000e+00
11            cartservice      socket            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         4.000000e+00         4.000000e+00  4.000000e+00  4.000000e+00
17        checkoutservice      socket            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         9.000000e+00         9.000000e+00  9.000000e+00  9.000000e+00
23        currencyservice      socket            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         4.000000e+00         4.000000e+00  4.000000e+00  4.000000e+00
26           emailservice      diskio            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
30           emailservice      socket            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         3.000000e+00         3.000000e+00  3.000000e+00  3.000000e+00
33               frontend       error            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
39      frontend-external       error            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
45         paymentservice      socket            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         3.000000e+00         3.000000e+00  3.000000e+00  3.000000e+00
54  recommendationservice      diskio            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
58  recommendationservice      socket            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         4.000000e+00         4.000000e+00  4.000000e+00  4.000000e+00
63                  redis      socket            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         4.000000e+00         4.000000e+00  4.000000e+00  4.000000e+00
68        shippingservice      socket            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         2.000000e+00         2.000000e+00  2.000000e+00  2.000000e+00
37               frontend      socket            25          22          2  2024-01-17T10:11:00+00:00  2024-01-17T10:08:00+00:00         2.580000e+01         2.400000e+01  2.500000e+01  2.401667e+01
61                  redis      diskio            25           2         19  2024-01-17T10:20:00+00:00  2024-01-17T10:08:00+00:00         2.254972e+04         0.000000e+00  2.188876e+04  0.000000e+00
51  productcatalogservice      socket            25          11         13  2024-01-17T10:22:00+00:00  2024-01-17T10:08:00+00:00         6.000000e+00         4.000000e+00  6.000000e+00  4.000000e+00
27           emailservice  latency-50            25           2         10  2024-01-17T10:25:00+00:00  2024-01-17T10:08:00+00:00         3.118178e-03         3.000000e-03  3.101784e-03  3.000000e-03
28           emailservice  latency-90            25           2         10  2024-01-17T10:25:00+00:00  2024-01-17T10:08:00+00:00         4.812721e-03         4.600000e-03  4.783212e-03  4.600000e-03
62                  redis         mem            25           2          3  2024-01-17T10:10:00+00:00  2024-01-17T10:08:00+00:00         3.039369e+06         2.960657e+06  3.027709e+06  2.961408e+06
0               adservice         cpu            25           2          2  2024-01-17T10:16:00+00:00  2024-01-17T10:24:00+00:00         8.569787e-01         5.794831e-01  7.568786e-01  6.215579e-01```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse existing trace_df if available; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
    trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Ensure timestamp and value dtypes
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, trace_name) using the entire trace series
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(global_P95=('value', lambda s: float(s.quantile(0.95)) if s.notna().any() else float('nan')))
)

# 2) Filter trace rows to the incident window (UTC inclusive)
start = pd.to_datetime('2024-01-17 10:05:48', utc=True)
end = pd.to_datetime('2024-01-17 10:35:48', utc=True)
trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# 3) Restrict to specified components (including frontend)
components = ['checkoutservice','currencyservice','emailservice','productcatalogservice','recommendationservice','frontend']
trace_window = trace_window[trace_window['cmdb_id'].isin(components)].copy()

# If no rows in window, prepare empty result
cols = ['cmdb_id','trace_name','window_count','high_count','earliest_high_ts','max_value_in_window','global_P95']
if trace_window.shape[0] == 0:
    trace_result = pd.DataFrame(columns=cols)
else:
    # Merge global P95 into the windowed data (thresholds computed globally)
    merged = trace_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')

    rows = []
    grouped = merged.groupby(['cmdb_id', 'trace_name'], sort=False)
    for (cmdb, trace_name), g in grouped:
        vals = g['value']
        gp95 = float(g['global_P95'].iloc[0])

        window_count = int(vals.count())
        high_mask = vals >= gp95
        high_count = int(high_mask.sum())

        if high_count > 0:
            earliest_high_ts = g.loc[high_mask, 'timestamp'].min().isoformat()
        else:
            earliest_high_ts = None

        max_value_in_window = float(vals.max()) if vals.notna().any() else float('nan')

        rows.append({
            'cmdb_id': cmdb,
            'trace_name': trace_name,
            'window_count': window_count,
            'high_count': high_count,
            'earliest_high_ts': earliest_high_ts,
            'max_value_in_window': max_value_in_window,
            'global_P95': gp95
        })

    trace_result = pd.DataFrame(rows)

    # 5) Keep only rows with window_count>0, sort by high_count desc then window_count desc, limit to top 20
    trace_result = trace_result[trace_result['window_count'] > 0].copy()
    trace_result = trace_result.sort_values(['high_count', 'window_count'], ascending=[False, False]).head(20)

# Display compact result
trace_result

```
Out[3]:
```
summary = (
    "Trace summary (checkoutservice, currencyservice, emailservice, productcatalogservice, "
    "recommendationservice, frontend) for window 2024-01-17 10:05:48 → 10:35:48 UTC:\n\n"
    "- For all listed trace.error_rate series the global P95 = 0.0 and every window sample = 0.0, "
    "so window_count=25 and high_count=25 (this is an artifact of threshold == observed value, not an alert). "
    "max_value_in_window is 0.0 and earliest_high_ts is 2024-01-17T10:08:00+00:00 for these series.\n\n"
    "- Interpretation: error_rate traces show no elevated errors (all zeros). The high_count results are not indicative "
    "of problems because the global P95 equals the observed values.\n\n"
    "- Recommended next steps: inspect non-error-rate trace metrics (duration mean/p95) and related logs/metrics for "
    "frontend and downstream services (e.g., frontend socket, productcatalogservice, redis diskio) identified earlier, "
    "since error_rate traces do not show anomalies."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   trace_name  window_count  high_count           earliest_high_ts  max_value_in_window  global_P95
2         checkoutservice        trace.from_checkoutservice.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
6         checkoutservice        trace.from_frontendservice.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
10        checkoutservice                   trace.from_root.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
14        checkoutservice          trace.to_checkoutservice.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
18        checkoutservice          trace.to_currencyservice.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
22        checkoutservice             trace.to_emailservice.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
26        checkoutservice           trace.to_paymentservice.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
30        checkoutservice    trace.to_productcatalogservice.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
34        currencyservice        trace.from_checkoutservice.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
38        currencyservice        trace.from_frontendservice.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
42        currencyservice                   trace.from_root.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
46           emailservice        trace.from_checkoutservice.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
50           emailservice                   trace.from_root.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
54  productcatalogservice        trace.from_checkoutservice.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
58  productcatalogservice        trace.from_frontendservice.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
62  productcatalogservice  trace.from_recommendationservice.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
66  productcatalogservice                   trace.from_root.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
70  recommendationservice        trace.from_frontendservice.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
74  recommendationservice  trace.from_recommendationservice.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0
78  recommendationservice                   trace.from_root.error_rate            25          25  2024-01-17T10:08:00+00:00                  0.0         0.0```
```

In [ ]:
# In[4]:

import pandas as pd
import re

# Reuse existing trace_df if available; otherwise load and prepare
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
    trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Ensure correct dtypes (idempotent)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, trace_name) using the entire trace series (global thresholds)
trace_duration_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(global_P95=('value', lambda s: float(s.quantile(0.95)) if s.notna().any() else float('nan')))
)

# 2) Filter trace rows to the incident window (UTC inclusive)
start = pd.to_datetime('2024-01-17 10:05:48', utc=True)
end = pd.to_datetime('2024-01-17 10:35:48', utc=True)
trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# 3) Restrict to specified components and duration-related trace_names
components = ['checkoutservice','currencyservice','emailservice','productcatalogservice','recommendationservice','frontend']
trace_window = trace_window[trace_window['cmdb_id'].isin(components)].copy()

# filter trace_name for duration metrics (case-insensitive)
trace_window = trace_window[trace_window['trace_name'].str.contains(r'duration|duration_mean|duration_p95', flags=re.IGNORECASE, regex=True)].copy()

# Prepare result columns
cols = ['cmdb_id','trace_name','window_count','high_count','earliest_high_ts','max_value_in_window','global_P95']

if trace_window.shape[0] == 0:
    trace_duration_result = pd.DataFrame(columns=cols)
else:
    # 4) Merge global P95 thresholds (computed globally) into windowed data
    merged = trace_window.merge(trace_duration_thresholds, on=['cmdb_id','trace_name'], how='left')

    rows = []
    grouped = merged.groupby(['cmdb_id','trace_name'], sort=False)
    for (cmdb, tname), g in grouped:
        vals = g['value']
        gp95 = float(g['global_P95'].iloc[0])

        window_count = int(vals.count())
        high_mask = vals >= gp95
        high_count = int(high_mask.sum())

        if high_count > 0:
            earliest_high_ts = g.loc[high_mask, 'timestamp'].min().isoformat()
        else:
            earliest_high_ts = None

        max_value_in_window = float(vals.max()) if vals.notna().any() else float('nan')

        rows.append({
            'cmdb_id': cmdb,
            'trace_name': tname,
            'window_count': window_count,
            'high_count': high_count,
            'earliest_high_ts': earliest_high_ts,
            'max_value_in_window': max_value_in_window,
            'global_P95': gp95
        })

    trace_duration_result = pd.DataFrame(rows)
    # 5) Keep only rows with window_count>0, sort by high_count desc then window_count desc, limit to top 20
    trace_duration_result = trace_duration_result[trace_duration_result['window_count'] > 0].copy()
    trace_duration_result = trace_duration_result.sort_values(['high_count','window_count'], ascending=[False, False]).head(20)

# Display compact result
trace_duration_result

```
Out[4]:
```
Summary of trace-duration findings (window 2024-01-17 10:05:48 → 10:35:48 UTC):

- The strongest signal is productcatalogservice — trace.from_frontendservice.duration_p95:
  - window_count=25, high_count=4, earliest_high=2024-01-17T10:21:00Z, max_in_window=0.000033 (global P95=0.000032).
  - Indicates multiple samples just above the global P95 around 10:21 UTC.

- Checkoutservice shows repeated small duration increases (each with high_count=2), many with earliest_high at 2024-01-17T10:21:00Z:
  - Examples: trace.from_checkoutservice.duration_mean/p95, trace.from_frontendservice.duration_mean/p95, trace.to_productcatalogservice.duration_mean/p95.
  - These point to elevated latencies around 10:21 for checkoutservice and its calls to productcatalogservice.

- Other notes:
  - Some shorter/earlier spikes exist (currencyservice durations show highs at ~10:14 and ~10:25; root/from_root durations show highs ~10:12).
  - Absolute duration values are small (sub-second), but they exceed their global P95 baselines.

Interpretation / next steps:
- The timing convergence (~10:21 UTC) and the involvement of checkoutservice ↔ productcatalogservice suggest a latency issue in the checkout → productcatalog call path (as observed from frontend traces).
- Investigate logs, metrics (socket, CPU, diskio) and traces for productcatalogservice and checkoutservice around 10:20–10:25 UTC (and frontend call patterns) to pinpoint root cause.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                    trace_name  window_count  high_count           earliest_high_ts  max_value_in_window  global_P95
29  productcatalogservice       trace.from_frontendservice.duration_p95            25           4  2024-01-17T10:21:00+00:00             0.000033    0.000032
0         checkoutservice      trace.from_checkoutservice.duration_mean            25           2  2024-01-17T10:21:00+00:00             0.024043    0.021261
1         checkoutservice       trace.from_checkoutservice.duration_p95            25           2  2024-01-17T10:21:00+00:00             0.081199    0.080320
2         checkoutservice      trace.from_frontendservice.duration_mean            25           2  2024-01-17T10:21:00+00:00             0.217350    0.204234
3         checkoutservice       trace.from_frontendservice.duration_p95            25           2  2024-01-17T10:21:00+00:00             0.913539    0.794436
4         checkoutservice                 trace.from_root.duration_mean            25           2  2024-01-17T10:12:00+00:00             0.016433    0.016078
5         checkoutservice                  trace.from_root.duration_p95            25           2  2024-01-17T10:12:00+00:00             0.073501    0.072962
6         checkoutservice        trace.to_checkoutservice.duration_mean            25           2  2024-01-17T10:21:00+00:00             0.024043    0.021261
7         checkoutservice         trace.to_checkoutservice.duration_p95            25           2  2024-01-17T10:21:00+00:00             0.081199    0.080320
8         checkoutservice        trace.to_currencyservice.duration_mean            25           2  2024-01-17T10:14:00+00:00             0.002298    0.001443
9         checkoutservice         trace.to_currencyservice.duration_p95            25           2  2024-01-17T10:25:00+00:00             0.000307    0.000256
10        checkoutservice           trace.to_emailservice.duration_mean            25           2  2024-01-17T10:17:00+00:00             0.000350    0.000320
11        checkoutservice            trace.to_emailservice.duration_p95            25           2  2024-01-17T10:17:00+00:00             0.000765    0.000543
12        checkoutservice         trace.to_paymentservice.duration_mean            25           2  2024-01-17T10:19:00+00:00             0.000440    0.000286
13        checkoutservice          trace.to_paymentservice.duration_p95            25           2  2024-01-17T10:19:00+00:00             0.001075    0.000828
14        checkoutservice  trace.to_productcatalogservice.duration_mean            25           2  2024-01-17T10:24:00+00:00             0.001220    0.001183
15        checkoutservice   trace.to_productcatalogservice.duration_p95            25           2  2024-01-17T10:24:00+00:00             0.000046    0.000037
16        currencyservice      trace.from_checkoutservice.duration_mean            25           2  2024-01-17T10:14:00+00:00             0.002298    0.001443
17        currencyservice       trace.from_checkoutservice.duration_p95            25           2  2024-01-17T10:25:00+00:00             0.000307    0.000256
18        currencyservice      trace.from_frontendservice.duration_mean            25           2  2024-01-17T10:08:00+00:00             0.000559    0.000514```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse existing dataframes if present, otherwise load
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
try:
    error_df
except NameError:
    error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes per rules
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Define incident window (UTC inclusive)
start = pd.to_datetime('2024-01-17 10:05:48', utc=True)
end = pd.to_datetime('2024-01-17 10:35:48', utc=True)

# Components of interest
components = ["productcatalogservice", "checkoutservice"]

# 1-2) Filter log rows to window and components
logs_win = log_df[
    (log_df['timestamp'] >= start) &
    (log_df['timestamp'] <= end) &
    (log_df['cmdb_id'].isin(components))
].copy()

# Prepare timestamp as ISO UTC strings for display
logs_win['timestamp'] = logs_win['timestamp'].apply(lambda ts: ts.isoformat())

# 3) Return up to 50 rows sorted by timestamp ascending with selected columns
logs_table = logs_win.sort_values('timestamp', ascending=True).loc[:, ['timestamp', 'cmdb_id', 'log_name', 'value']].head(50)

# Aggregate counts per (cmdb_id, log_name) within the window (limit to 20 rows)
logs_agg = (
    logs_win
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(count=('value', 'count'))
    .sort_values('count', ascending=False)
    .head(20)
)

# 4) Filter error_logs to window and components, sort earliest, return up to 20 rows
errors_win = error_df[
    (error_df['timestamp'] >= start) &
    (error_df['timestamp'] <= end) &
    (error_df['cmdb_id'].isin(components))
].copy()

errors_win['timestamp'] = errors_win['timestamp'].apply(lambda ts: ts.isoformat())
error_table = errors_win.sort_values('timestamp', ascending=True).loc[:, ['timestamp', 'cmdb_id', 'message']].head(20)

# Display compact outputs
logs_table, logs_agg, error_table

```
Out[5]:
```
Summary for the incident window (2024-01-17 10:05:48 → 10:35:48 UTC), focused on productcatalogservice and checkoutservice:

- Logs (log.csv):
  - Only checkoutservice produced log rows in the window. We returned 50 ascending rows (sample shown) and an aggregate:
    - checkoutservice — log.error_count: 25 rows, all values = 0 (no logged errors).
    - checkoutservice — log.row_count: 25 rows, varying counts (examples in the window: 9, 78, up to ~117, etc.), indicating normal/variable request activity.
  - productcatalogservice had no log entries in this window (no rows returned).

- Error logs (error_logs.csv):
  - No error_log rows were found for checkoutservice or productcatalogservice within the incident window (empty result).

Interpretation / recommended next steps:
- There are no logged errors for checkoutservice or productcatalogservice in the window, so investigate other telemetry signals already highlighted (frontend socket spikes, checkout → productcatalog latency around ~10:21 UTC, and redis diskio spikes) and check traces around the earliest_high timestamps to pinpoint root cause.
- If you want, I can show more of the checkoutservice log.row_count time series around 10:20–10:25 UTC or search other services' logs/traces for corroborating evidence.

The original code execution output of IPython Kernel is also provided below for reference:

(                     timestamp          cmdb_id         log_name  value
4    2024-01-17T10:08:00+00:00  checkoutservice  log.error_count      0
5    2024-01-17T10:08:00+00:00  checkoutservice    log.row_count      9
22   2024-01-17T10:09:00+00:00  checkoutservice  log.error_count      0
23   2024-01-17T10:09:00+00:00  checkoutservice    log.row_count     78
42   2024-01-17T10:10:00+00:00  checkoutservice  log.error_count      0
..                         ...              ...              ...    ...
409  2024-01-17T10:30:00+00:00  checkoutservice    log.row_count    117
428  2024-01-17T10:31:00+00:00  checkoutservice  log.error_count      0
429  2024-01-17T10:31:00+00:00  checkoutservice    log.row_count     66
446  2024-01-17T10:32:00+00:00  checkoutservice  log.error_count      0
447  2024-01-17T10:32:00+00:00  checkoutservice    log.row_count     51

[50 rows x 4 columns],            cmdb_id         log_name  count
0  checkoutservice  log.error_count     25
1  checkoutservice    log.row_count     25, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse existing metric_df if available; otherwise load and prepare
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp and value dtypes per rules (timestamps are Unix seconds -> UTC)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Allowed components
allowed = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using entire series (for allowed components)
thresholds = (
    metric_df[metric_df['cmdb_id'].isin(allowed)]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(global_P95=('value', lambda s: float(s.quantile(0.95)) if s.notna().any() else float('nan')),
         global_P05=('value', lambda s: float(s.quantile(0.05)) if s.notna().any() else float('nan')))
)

# 2) Filter metric rows to the incident window (UTC inclusive)
start = pd.to_datetime('2024-01-17 10:05:48', utc=True)
end = pd.to_datetime('2024-01-17 10:35:48', utc=True)

window_df = metric_df[
    (metric_df['timestamp'] >= start) &
    (metric_df['timestamp'] <= end) &
    (metric_df['cmdb_id'].isin(allowed))
].copy()

# Helper to compute longest consecutive True run for a boolean Series (ordered by index)
def longest_consecutive_run_bool(s: pd.Series) -> int:
    if s.empty:
        return 0
    arr = s.astype(bool).to_numpy()
    # compute run lengths
    # fast vectorized approach
    # when arr[i] == True, it's part of a run; compute lengths by diff of boundaries
    # fallback to simple loop for clarity and reliability
    max_run = 0
    cur = 0
    for v in arr:
        if v:
            cur += 1
            if cur > max_run:
                max_run = cur
        else:
            cur = 0
    return int(max_run)

# 3) For each (cmdb_id, kpi_name) within the window compute required stats
rows = []
if not window_df.empty:
    # merge global thresholds into window rows for comparisons (thresholds computed globally)
    merged = window_df.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

    # group by series
    grouped = merged.groupby(['cmdb_id', 'kpi_name'], sort=False)
    for (cmdb, kpi), g in grouped:
        g_sorted = g.sort_values('timestamp')  # ensure chronological order for run detection
        vals = g_sorted['value']
        gp95 = float(g_sorted['global_P95'].iloc[0]) if pd.notna(g_sorted['global_P95'].iloc[0]) else float('nan')
        gp05 = float(g_sorted['global_P05'].iloc[0]) if pd.notna(g_sorted['global_P05'].iloc[0]) else float('nan')

        window_count = int(vals.count())
        high_mask = vals >= gp95
        low_mask = vals <= gp05
        high_count = int(high_mask.sum())
        low_count = int(low_mask.sum())

        earliest_high_ts = g_sorted.loc[high_mask, 'timestamp'].min().isoformat() if high_count > 0 else None
        earliest_low_ts = g_sorted.loc[low_mask, 'timestamp'].min().isoformat() if low_count > 0 else None

        max_value_in_window = float(vals.max()) if vals.notna().any() else float('nan')
        min_value_in_window = float(vals.min()) if vals.notna().any() else float('nan')

        longest_high_run = longest_consecutive_run_bool(high_mask)
        longest_low_run = longest_consecutive_run_bool(low_mask)

        rows.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'window_count': window_count,
            'high_count': high_count,
            'low_count': low_count,
            'earliest_high_ts': earliest_high_ts,
            'earliest_low_ts': earliest_low_ts,
            'max_value_in_window': max_value_in_window,
            'min_value_in_window': min_value_in_window,
            'global_P95': gp95,
            'global_P05': gp05,
            'longest_consecutive_high_run': longest_high_run,
            'longest_consecutive_low_run': longest_low_run
        })

# assemble result DataFrame
scan_result = pd.DataFrame(rows)

# 4) Filter to window_count>0 and sort by longest run severity then high_count, limit to top 20
if not scan_result.empty:
    scan_result['max_long_run'] = scan_result[['longest_consecutive_high_run', 'longest_consecutive_low_run']].max(axis=1)
    scan_result = (
        scan_result[scan_result['window_count'] > 0]
        .sort_values(['max_long_run', 'high_count'], ascending=[False, False])
        .drop(columns=['max_long_run'])
        .reset_index(drop=True)
    )
    scan_result = scan_result.head(20)

# Display compact final table (top candidates)
scan_result

```
Out[6]:
```
Summary of metric fault-scan for allowed components (window 2024-01-17 10:05:48 → 10:35:48 UTC):

- Many series (notably several "socket" and some "diskio") are constant in the window so global P05 == P95 == observed value. Those show high_count = low_count = window_count and longest consecutive runs = 25 — this is a data artifact (threshold == value) and not a useful anomaly by itself.

- Most actionable signals:
  - productcatalogservice — socket:
    - window_count=25, high_count=11, low_count=13
    - earliest_high=2024-01-17T10:22:00Z, earliest_low=2024-01-17T10:08:00Z
    - max=6, min=4, longest consecutive high_run=11 / low_run=13
    - Interpretation: mixed behavior with sustained low and later sustained higher socket values — worth investigating (timing aligns with trace-duration spikes ~10:21).
  - checkoutservice — latency (latency-50, latency-90):
    - small number of high samples (high_count=2) with earliest highs ~10:22–10:23 UTC and elevated max values vs baseline — indicates short-lived latency increases in checkout path.
  - emailservice — latency-50/90:
    - high_count=2 with earliest_high around 2024-01-17T10:25:00Z (many low samples earlier) — short latency elevations.
  - Some mem/cpu series show modest excursions (e.g., productcatalogservice mem and currencyservice mem/cpu) with small high_run lengths (2), indicating brief resource changes rather than long outages.

- Priority candidates to investigate now:
  1. productcatalogservice (socket behavior mixed, sustained runs and timing ~10:22).
  2. checkoutservice (latency spikes around ~10:21–10:24; part of the checkout → productcatalog call path).
  3. emailservice (short latency elevations ~10:25) as a secondary signal.

- Recommended next steps:
  - Inspect traces and logs for checkoutservice ↔ productcatalogservice around 10:20–10:25 UTC (earliest_high timestamps) — correlate with frontend trace spikes already observed.
  - Ignore constant-series artifacts (socket/diskio with identical P05/P95) unless additional context (logs/traces) indicates otherwise.
  - Check related infra telemetry (network/socket, redis/diskio) around the same timestamps for supporting evidence.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  window_count  high_count  low_count           earliest_high_ts            earliest_low_ts  max_value_in_window  min_value_in_window    global_P95    global_P05  longest_consecutive_high_run  longest_consecutive_low_run
0         checkoutservice      socket            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         9.000000e+00         9.000000e+00  9.000000e+00  9.000000e+00                            25                           25
1         currencyservice      socket            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         4.000000e+00         4.000000e+00  4.000000e+00  4.000000e+00                            25                           25
2            emailservice      diskio            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00                            25                           25
3            emailservice      socket            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         3.000000e+00         3.000000e+00  3.000000e+00  3.000000e+00                            25                           25
4   recommendationservice      diskio            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00                            25                           25
5   recommendationservice      socket            25          25         25  2024-01-17T10:08:00+00:00  2024-01-17T10:08:00+00:00         4.000000e+00         4.000000e+00  4.000000e+00  4.000000e+00                            25                           25
6   productcatalogservice      socket            25          11         13  2024-01-17T10:22:00+00:00  2024-01-17T10:08:00+00:00         6.000000e+00         4.000000e+00  6.000000e+00  4.000000e+00                            11                           13
7            emailservice  latency-50            25           2         10  2024-01-17T10:25:00+00:00  2024-01-17T10:08:00+00:00         3.118178e-03         3.000000e-03  3.101784e-03  3.000000e-03                             1                            5
8            emailservice  latency-90            25           2         10  2024-01-17T10:25:00+00:00  2024-01-17T10:08:00+00:00         4.812721e-03         4.600000e-03  4.783212e-03  4.600000e-03                             1                            5
9         checkoutservice  latency-50            25           2          2  2024-01-17T10:23:00+00:00  2024-01-17T10:09:00+00:00         1.819149e-01         8.694592e-02  1.805951e-01  9.022369e-02                             1                            2
10        checkoutservice  latency-90            25           2          2  2024-01-17T10:22:00+00:00  2024-01-17T10:09:00+00:00         5.104858e-01         2.148067e-01  4.106576e-01  2.211523e-01                             1                            2
11        currencyservice  latency-90            25           2          2  2024-01-17T10:10:00+00:00  2024-01-17T10:26:00+00:00         4.538497e-02         9.436417e-03  4.083674e-02  9.478988e-03                             1                            2
12        currencyservice         mem            25           2          2  2024-01-17T10:19:00+00:00  2024-01-17T10:29:00+00:00         4.483297e+07         4.398845e+07  4.476066e+07  4.400488e+07                             2                            2
13           emailservice         mem            25           2          2  2024-01-17T10:31:00+00:00  2024-01-17T10:08:00+00:00         4.342170e+07         4.339623e+07  4.341974e+07  4.340030e+07                             2                            1
14  productcatalogservice         mem            25           2          2  2024-01-17T10:23:00+00:00  2024-01-17T10:08:00+00:00         1.534020e+07         1.001779e+07  1.526548e+07  1.014433e+07                             2                            1
15  recommendationservice  latency-50            25           2          2  2024-01-17T10:28:00+00:00  2024-01-17T10:17:00+00:00         6.070747e-02         7.425702e-03  5.796952e-02  7.434110e-03                             1                            2
16        checkoutservice         cpu            25           2          2  2024-01-17T10:24:00+00:00  2024-01-17T10:16:00+00:00         5.490787e-01         3.029639e-01  5.288097e-01  3.213399e-01                             1                            1
17        checkoutservice         mem            25           2          2  2024-01-17T10:09:00+00:00  2024-01-17T10:13:00+00:00         1.181191e+07         1.119840e+07  1.170903e+07  1.120894e+07                             1                            1
18        checkoutservice    workload            25           2          2  2024-01-17T10:24:00+00:00  2024-01-17T10:16:00+00:00         1.353267e+00         5.888667e-01  1.090190e+00  6.463433e-01                             1                            1
19        currencyservice         cpu            25           2          2  2024-01-17T10:13:00+00:00  2024-01-17T10:30:00+00:00         3.332740e+01         2.715515e+01  3.275977e+01  2.718900e+01                             1                            1```
```